In [27]:
import numpy as np
import torch

In [28]:
class Config:
    device = "cuda" if torch.cuda.is_available() else "cpu"

In [29]:
class FromScratchCNN:
    def __init__(self, input_shape, kernel_sizes, dense_layers, strides=1, padding='valid'):
        self.strides = strides
        self.padding = padding
        self.kernel_sizes = kernel_sizes
        
        # 1. Calculate the final flattened size after convolutions
        h, w = input_shape[-2], input_shape[-1]
        for k in kernel_sizes:
            h, w = h - k + 1, w - k + 1
            
        # 2. Initialize CNN Kernels 
        # Crucial: We do NOT set requires_grad=True because we are writing our own backprop!
        self.kernels = [
            torch.randn(k, k, device=Config.device) * 0.1 for k in kernel_sizes
        ]
        
        # 3. Initialize Dense Layer Weights
        # Flattened input size -> hidden layer -> output layer
        self.dense_weights = [
            torch.randn(h * w, dense_layers[0], device=Config.device) * 0.1,
            torch.randn(dense_layers[0], 1, device=Config.device) * 0.1
        ]

        self.biases = [
            torch.zeros(dense_layers[0], device=Config.device),
            torch.zeros(1, device=Config.device)
        ]

        # Tracking lists for backpropagation
        self.c_inputs = []
        self.c_activations = []
        self.d_inputs = []
        self.d_z_values = []

    def relu(self, x):
        return torch.clamp(x, min=0)

    def relu_derivative(self, x):
        return torch.where(x > 0, 1.0, 0.0)

    # ==========================================
    # MANUALLY BUILDING THE SLIDING WINDOW
    # ==========================================
    def convolve(self, X, kernel):
        # Expecting single 2D image matrix [Height, Width]
        height, width = X.shape
        k_h, k_w = kernel.shape

        out_height = (height - k_h) // self.strides + 1
        out_width = (width - k_w) // self.strides + 1
        
        # Creating a safe tensor without breaking the computational graph
        output_map = torch.zeros((out_height, out_width), device=Config.device)

        for i in range(out_height):
            for j in range(out_width):
                h_start = i * self.strides
                w_start = j * self.strides
                
                region = X[h_start : h_start + k_h, w_start : w_start + k_w]
                # Scalar accumulation at position [i, j]
                output_map[i, j] = torch.sum(region * kernel)

        return output_map

    # ==========================================
    # THE COMPLETE FORWARD PASS
    # ==========================================
    def forward(self, X, y_true):
        self.c_inputs = []
        self.c_activations = []
        self.d_inputs = []
        self.d_z_values = []
        
        # 1. CNN Feature Extraction
        current_input = X
        for kernel in self.kernels:
            self.c_inputs.append(current_input)
            conv_out = self.convolve(current_input, kernel)
            current_input = self.relu(conv_out)
            self.c_activations.append(current_input)
            
        # 2. Flattening Step
        flattened = current_input.view(1, -1) 
        
        # 3. Dense Classification Loop
        current_dense_input = flattened
        for i, W in enumerate(self.dense_weights):
            self.d_inputs.append(current_dense_input)
            z = torch.mm(current_dense_input, W) + self.biases[i]
            self.d_z_values.append(z)
            
            if i < len(self.dense_weights) - 1:
                current_dense_input = self.relu(z)
            else:
                current_dense_input = z # Linear output layer
                
        pred = current_dense_input
        loss = torch.mean((y_true - pred) ** 2)
        return pred, loss

    # ==========================================
    # THE PURE FROM-SCRATCH BACKPROPAGATION
    # ==========================================
    def backward(self, y_true, learning_rate):
        # --- PHASE 1: Dense Network Backprop ---
        # Initial MSE Loss derivative w.r.t final prediction
        delta = 2.0 * (self.d_z_values[-1] - y_true)
        
        for i in reversed(range(len(self.dense_weights))):
            grad_w = torch.mm(self.d_inputs[i].t(), delta)
            grad_b = delta.mean(dim=0)  # Average over batch size
            if i > 0:
                prev_z = self.d_z_values[i-1]
                delta = torch.mm(delta, self.dense_weights[i].t()) * self.relu_derivative(prev_z)
                
            # Manual weight update
            self.dense_weights[i] -= learning_rate * grad_w
            self.biases[i] -= learning_rate * grad_b
            
        # The Bridge: Calculate error leaving the dense layer and entering the flatten layer
        grad_at_flatten = torch.mm(delta, self.dense_weights[0].t())
        
        # --- PHASE 2: CNN Layer Backprop (Your Exact Handwritten Math) ---
        # Un-flatten the 1D error back into the 2D matrix shape
        grad = grad_at_flatten.view(self.c_activations[-1].shape)
        
        for i in reversed(range(len(self.kernels))):
            layer_input = self.c_inputs[i]
            kernel = self.kernels[i]
            grad = grad * self.relu_derivative(self.c_activations[i])
            # STEP A: Weight Gradient (Matrix * Scalar Accumulation)
            grad_k = torch.zeros_like(kernel, device=Config.device)
            for m in range(grad.shape[0]):
                for n in range(grad.shape[1]):
                    region = layer_input[m : m + kernel.shape[0], n : n + kernel.shape[1]]
                    grad_k += region * grad[m, n]
            
            # STEP B: Input Gradient (Matrix * Matrix -> Sum to Scalar)
            # flipped_kernel = torch.tensor(np.rot90(kernel.cpu().numpy(), 2), device=Config.device)
            flipped_kernel = torch.tensor(np.rot90(kernel.cpu().numpy(), 2).copy(), device=Config.device)
            pad_h, pad_w = kernel.shape[0] - 1, kernel.shape[1] - 1
            padded_grad = torch.nn.functional.pad(grad, (pad_w, pad_w, pad_h, pad_h), mode='constant', value=0)
            
            next_grad = torch.zeros_like(layer_input, device=Config.device)
            for m in range(next_grad.shape[0]):
                for n in range(next_grad.shape[1]):
                    region = padded_grad[m : m + flipped_kernel.shape[0], n : n + flipped_kernel.shape[1]]
                    next_grad[m, n] = torch.sum(region * flipped_kernel)
            
            # Update the kernel weights completely manually
            self.kernels[i] -= learning_rate * grad_k
            grad = next_grad # Error flows backward to previous layer

    def train(self, X, y, learning_rate=0.01, epochs=100):
        for epoch in range(epochs):
            pred, loss = self.forward(X, y)
            self.backward(y, learning_rate)
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

    def predict(self, X):
        current_input = X
        for kernel in self.kernels:
            conv_out = self.convolve(current_input, kernel)
            current_input = self.relu(conv_out)

        flattened = current_input.view(1, -1)
        current_dense_input = flattened
        for i, W in enumerate(self.dense_weights):
            z = torch.mm(current_dense_input, W)
            if i < len(self.dense_weights) - 1:
                current_dense_input = self.relu(z)
            else:
                current_dense_input = z # Linear output layer
        return current_dense_input


In [30]:
mock_image = torch.tensor([[1,0,0,0,0],[0,1,0,0,0],[0,0,1,0,0],[0,0,0,1,0],[0,0,0,0,1]], dtype=torch.float32, device=Config.device)
mock_target = torch.tensor([[1.0]], dtype=torch.float32, device=Config.device)
scratch_net = FromScratchCNN(input_shape=(5, 5), kernel_sizes=[3], dense_layers=[10])
mock_target = mock_target.view(1, -1)  # Ensure target is in the correct shape for training
scratch_net.train(mock_image, mock_target, learning_rate=0.01, epochs=100)

Epoch 10/100, Loss: 0.6821
Epoch 20/100, Loss: 0.4450
Epoch 30/100, Loss: 0.2892
Epoch 40/100, Loss: 0.1872
Epoch 50/100, Loss: 0.1206
Epoch 60/100, Loss: 0.0773
Epoch 70/100, Loss: 0.0494
Epoch 80/100, Loss: 0.0314
Epoch 90/100, Loss: 0.0199
Epoch 100/100, Loss: 0.0126
